In [ ]:
# MATLAB section 1/1 for FitResSummaryExamples: FitResSummary Examples

# % FitResSummary Examples

# Python translation bootstrap + execution for single-section helpfile.

# Topic: FitResSummaryExamples
# Execution group: full
# Workflow family: analysis
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/FitResSummaryExamples.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "FitResSummaryExamples"
FAMILY = "analysis"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"FitResSummaryExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"FitResSummaryExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"FitResSummaryExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"FitResSummaryExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


# FitResSummaryExamples: compare multiple fit results with IC summaries.
from nstat.compat.matlab import Analysis, FitResSummary

dt = 0.01
t = np.arange(0.0, 10.0, dt)
x1 = np.sin(2.0 * np.pi * 0.6 * t)
x2 = np.cos(2.0 * np.pi * 0.2 * t + 0.15)
x3 = np.sin(2.0 * np.pi * 0.05 * t + 0.2)
eta = -2.2 + 0.7 * x1 - 0.5 * x2 + 0.3 * x3
y = rng.poisson(np.exp(eta) * dt)

fit1 = Analysis.fitGLM(X=np.column_stack([x1]), y=y, fitType="poisson", dt=dt)
fit2 = Analysis.fitGLM(X=np.column_stack([x1, x2]), y=y, fitType="poisson", dt=dt)
fit3 = Analysis.fitGLM(X=np.column_stack([x1, x2, x3]), y=y, fitType="poisson", dt=dt)

summary = FitResSummary([fit1, fit2, fit3])
best_aic = summary.bestByAIC()
best_bic = summary.bestByBIC()
diff_aic = summary.getDiffAIC()
diff_bic = summary.getDiffBIC()

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))
plt.sca(axes[0])
summary.plotAIC()
axes[0].set_title(f"{TOPIC}: AIC")
axes[0].set_xlabel("model index")
axes[0].set_ylabel("AIC")
plt.sca(axes[1])
summary.plotBIC()
axes[1].set_title("BIC")
axes[1].set_xlabel("model index")
axes[1].set_ylabel("BIC")
plt.tight_layout()
plt.show()

assert diff_aic.size == diff_bic.size and diff_aic.size > 0
assert np.isfinite(best_aic.aic()) and np.isfinite(best_bic.bic())

CHECKPOINT_METRICS = {
    "num_models": float(diff_aic.size),
    "best_aic_diff": float(np.min(diff_aic)),
    "best_bic_diff": float(np.min(diff_bic)),
}
CHECKPOINT_LIMITS = {
    "num_models": (3.0, 3.0),
    "best_aic_diff": (-10.0, 10.0),
    "best_bic_diff": (-10.0, 10.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
